In [12]:
import pandas as pd
import numpy as np
import os
data_folder = 'data'

In [13]:
csv_filename = f"all.csv"
try:
    df = pd.read_csv(os.path.join(data_folder, csv_filename))
except FileNotFoundError:
    print(f"File not found: {csv_filename}. ")

In [14]:
print(f"Loaded raw data with {len(df)} rows.")

Loaded raw data with 8000 rows.


In [15]:
# 检查 'cell_id' 是否存在
if 'cell_id' not in df.columns:
    raise ValueError(f"Error: 'cell_id' column not found in {os.path.join(data_folder, csv_filename)}.")
else:
    print("Found 'cell_id' column. Proceeding with aggregation.")

Found 'cell_id' column. Proceeding with aggregation.


In [16]:
feature_cols = ['height', 'roughness', 'length', 'wideth', 'adhesion', 'elastic_modulus']

In [17]:
# 确保所有特征列都存在
missing_cols = [col for col in feature_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Error: Missing feature columns in data: {missing_cols}")

In [18]:
# --- 关键步骤：构建 Aggregation 字典 (修复错误) ---
    
# Define the statistics to calculate for each feature
agg_funcs = ['mean', 'std', 'median']
    
# 1. Initialize a single, complete aggregation dictionary
agg_dict = {}
    
# 2. Add the 6 feature columns with their list of aggregation functions
for feature in feature_cols:
    agg_dict[feature] = agg_funcs
        
# 3. Add the non-numeric column 'cycle', instructing pandas to take the first value
# This assumes one cell_id corresponds to only one cycle, which is true for your data.
agg_dict['cycle'] = 'first'
    
print(f"Aggregating {len(feature_cols)} features by 'cell_id'...")
    
# --- 执行 Aggregation ---
    
# Group by 'cell_id' and apply the defined aggregation dictionary
df_agg = df.groupby('cell_id').agg(agg_dict).reset_index()

Aggregating 6 features by 'cell_id'...


In [19]:
df_agg.items

<bound method DataFrame.items of     cell_id   height                   roughness                      \
                mean       std  median      mean        std   median   
0         0  3.39325  0.042109  3.4100  203.1550  10.012491  202.400   
1         1  3.31920  0.070333  3.3330  233.6100  29.824167  239.200   
2         2  2.30210  0.033298  2.3090  180.3965  13.908601  183.490   
3         3  2.90625  0.046476  2.9280  168.7005  22.168496  164.900   
4         4  2.73455  0.057433  2.7430  172.0300  17.960808  164.800   
..      ...      ...       ...     ...       ...        ...      ...   
395     395  2.10725  0.058006  2.1100  154.3000  10.408296  155.100   
396     396  2.35690  0.052401  2.3765  119.4550  14.860420  116.450   
397     397  2.42870  0.078979  2.4490  133.1600  16.560587  131.600   
398     398  2.90395  0.049459  2.9250  136.1550   7.183422  137.150   
399     399  1.92035  0.023107  1.9230   87.6400   6.298077   85.985   

       length                 

In [22]:
new_columns = ['cell_id'] # 强制第一个列名为 'cell_id'
for col in df_agg.columns[1:]:
    if isinstance(col, tuple):
        # 这是一个 MultiIndex 列 (例如: ('height', 'mean'), ('cycle', 'first'))
        
        # 1. 检查是否为 'cycle' 列的聚合结果
        if col[1] == 'first':
            # 将 ('cycle', 'first') 命名为 'cycle'
            new_columns.append(col[0])
        else:
            # 将 ('feature', 'stat') 命名为 'feature_stat'
            new_columns.append(f"{col[0]}_{col[1]}")
    else:
        # 这是一个单字符串列 (例如: 'cell_id'，因为它来自 reset_index())
        #   保持原样，防止被错误命名为 cell_id_
        new_columns.append(col)

# 应用新的列名
if len(new_columns) == len(df_agg.columns):
    df_agg.columns = new_columns
    print(f"Successfully renamed all {len(new_columns)} columns.")
else:
    # 如果仍然出现长度不匹配，则立即报错，并打印详细信息以供调试
    raise ValueError(
        f"Column renaming failed: Expected {len(df_agg.columns)} columns, "
        f"but generated {len(new_columns)} new names."
        f"Existing columns: {df_agg.columns.tolist()}"
        f"Generated names: {new_columns}"
    )

df_agg.items

Successfully renamed all 20 columns.


<bound method DataFrame.items of      cell_id  height_mean  height_std  height_median  roughness_mean  \
0          0      3.39325    0.042109         3.4100        203.1550   
1          1      3.31920    0.070333         3.3330        233.6100   
2          2      2.30210    0.033298         2.3090        180.3965   
3          3      2.90625    0.046476         2.9280        168.7005   
4          4      2.73455    0.057433         2.7430        172.0300   
..       ...          ...         ...            ...             ...   
395      395      2.10725    0.058006         2.1100        154.3000   
396      396      2.35690    0.052401         2.3765        119.4550   
397      397      2.42870    0.078979         2.4490        133.1600   
398      398      2.90395    0.049459         2.9250        136.1550   
399      399      1.92035    0.023107         1.9230         87.6400   

     roughness_std  roughness_median  length_mean  length_std  length_median  \
0        10.012491    

In [23]:
# --- Final Steps ---
output_csv_filename = f"agg.csv"
output_csv_path = os.path.join(data_folder, output_csv_filename)
# Re-order columns for clarity
agg_feature_cols = [col for col in df_agg.columns if col not in ['cell_id', 'cycle']]
final_cols = ['cell_id', 'cycle'] + agg_feature_cols
df_agg = df_agg[final_cols]
    
print(f"Aggregation complete. Final shape: {df_agg.shape}") 
    
df_agg.to_csv(output_csv_path, index=False)
print(f"Aggregated cell data saved to '{output_csv_path}'.")

Aggregation complete. Final shape: (400, 20)
Aggregated cell data saved to 'data\agg.csv'.
